### Statistical Tests

In [4]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine, text
import os

In [5]:
load_dotenv()
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}"
    f"@{os.getenv('MYSQL_HOST')}/{os.getenv('MYSQL_DATABASE')}"
)

In [7]:
query = """
SELECT delay_minutes, is_monsoon, station_no, is_extreme_delay
FROM delays
"""
df = pd.read_sql(text(query), engine)
df.shape

(262256, 4)

### Monsoon vs Non-Monsoon Delay (t-test)

In [8]:
from scipy import stats

monsoon = df.loc[df["is_monsoon"] == 1, "delay_minutes"]
non_monsoon = df.loc[df["is_monsoon"] == 0, "delay_minutes"]

t_stat, p_val = stats.ttest_ind(monsoon, non_monsoon, equal_var=False)

In [9]:
print(f"Monsoon mean delay: {monsoon.mean():.2f} min (n={len(monsoon)})")
print(f"Non-Monsoon mean delay: {non_monsoon.mean():.2f} min (n={len(non_monsoon)})")
print(f"t-statistic: {t_stat:.3f}, p-value: {p_val:.5f}")

Monsoon mean delay: 37.28 min (n=87165)
Non-Monsoon mean delay: 38.89 min (n=175091)
t-statistic: -7.688, p-value: 0.00000


In [11]:
def cohens_d(a, b):
    n1, n2 = len(a), len(b)
    pooled_std = (((n1 - 1) * a.std()**2 + (n2 - 1) * b.std()**2) / (n1 + n2 - 2)) ** 0.5
    return (a.mean() - b.mean()) / pooled_std

print(f"Cohen's d: {cohens_d(monsoon, non_monsoon):.3f}")

Cohen's d: -0.032


Monsoon vs. Non-Monsoon delays differ by ~1.6 min (p < 0.001, Cohen’s d = -0.032) → statistically significant but practically trivial.


### Top-delay stations vs the rest (ANOVA)

In [12]:
top_stations_query = """
SELECT station_no, AVG(delay_minutes) AS avg_delay, COUNT(*) AS n_records
FROM delays
GROUP BY station_no
ORDER BY avg_delay DESC
LIMIT 5
"""
top_stations = pd.read_sql(text(top_stations_query), engine)
top_stations

,station_no,avg_delay,n_records
0,39,55.60692,1618
1,28,54.19716,2607
2,45,54.05922,1199
3,34,52.89626,2034
4,38,52.81606,1669


In [13]:
top_station_codes = top_stations["station_no"].tolist()

groups = [
    df.loc[df["station_no"] == s, "delay_minutes"]
    for s in top_station_codes
]

f_stat, p_val = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.3f}, p-value: {p_val:.5f}")

F-statistic: 0.784, p-value: 0.53549


In [14]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

subset = df[df["station_no"].isin(top_station_codes)]
tukey = pairwise_tukeyhsd(
    endog=subset["delay_minutes"],
    groups=subset["station_no"],
    alpha=0.05,
)
print(tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
    28     34  -1.3009 0.9241 -5.6215 3.0197  False
    28     38  -1.3811 0.9236 -5.9594 3.1972  False
    28     39   1.4098 0.9207 -3.2123 6.0318  False
    28     45  -0.1379    1.0  -5.234 4.9581  False
    34     38  -0.0802    1.0 -4.9036 4.7432  False
    34     39   2.7107 0.5493 -2.1543 7.5756  False
    34     45    1.163 0.9756 -4.1544 6.4803  False
    38     39   2.7909 0.5661 -2.3043 7.8861  False
    38     45   1.2432  0.973 -4.2857  6.772  False
    39     45  -1.5477 0.9423 -7.1128 4.0174  False
---------------------------------------------------


### Day of Week test (ANOVA)

In [16]:
dow_query = "SELECT delay_minutes, day_of_week FROM delays;"
df_dow = pd.read_sql(text(dow_query), engine)

dow_groups = [
    df_dow.loc[df_dow["day_of_week"] == d, "delay_minutes"]
    for d in df_dow["day_of_week"].unique()
]
f_stat, p_val = stats.f_oneway(*dow_groups)
print(f"Day of Week F-statistic: {f_stat:.3f}, p-value: {p_val:.5f}")

Day of Week F-statistic: 61.537, p-value: 0.00000


### Interpretation:
Monsoon vs Non-Monsoon: Statistically significant (p<0.001) but practically meaningless — Cohen's d = -0.032, only ~1.6 min difference. Monsoon is not a real driver of delay.   
Top-5 Delay Stations: ANOVA not significant (p=0.535), Tukey confirms no pairwise differences. The "worst" stations aren't actually distinguishable — likely noise.  
Day of Week: Strongly significant (F=61.5, p<0.001) — delay genuinely varies by day, the strongest real finding among the three tests.